# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        # Estanterías / paredes
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso (celdas amarillas)
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Terminales: entrega +10, carga +2, peligro mortal -10
        self.terminal_states = {
            (0, 5): 10.0,
            (2, 2): 2.0,
            (3, 5): -10.0,
        }

        # Peligros NO terminales -3
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _next_state(self, state, action):
        ns = (state[0] + action[0], state[1] + action[1])
        if not self.is_valid_state(ns):
            return state
        return ns

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        left = (-action[1], action[0])
        right = (action[1], -action[0])

        if state in self.slippery_states:
            probs = [(action, 0.60), (left, 0.20), (right, 0.20)]
        else:
            probs = [(action, 0.90), (left, 0.05), (right, 0.05)]

        outcomes = {}
        for act, p in probs:
            ns = self._next_state(state, act)
            outcomes[ns] = outcomes.get(ns, 0.0) + p

        return list(outcomes.items())
    


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    total = 0.0
    for next_state, prob in grid.get_transition_probs(state, action):
        total += prob * V[next_state]
    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0.0
        newV = V.copy()
        for s in grid.states():
            if grid.is_terminal(s):
                newV[s] = grid.get_reward(s)
                continue
            q_values = [expected_next_value(grid, s, a, V) for a in grid.actions]
            newV[s] = grid.get_reward(s) + grid.gamma * max(q_values)
            delta = max(delta, abs(newV[s] - V[s]))
        V = newV
        if delta < threshold:
            return V, i + 1
    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = [(a, expected_next_value(grid, s, a, V)) for a in grid.actions]
        policy[s] = max(q_values, key=lambda x: x[1])[0]
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0.0
        newV = V.copy()
        for s in grid.states():
            if grid.is_terminal(s):
                newV[s] = grid.get_reward(s)
                continue
            a = policy[s]
            newV[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(newV[s] - V[s]))
        V = newV
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = [(a, expected_next_value(grid, s, a, V)) for a in grid.actions]
        policy[s] = max(q_values, key=lambda x: x[1])[0]
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for i in range(max_iter):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)
        changes = sum(1 for s in new_policy if new_policy[s] != policy[s])
        history.append(changes)
        policy = new_policy
        if changes == 0:
            break
    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [17, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

Respuestas

**1.** Desde START, el robot prefiere ir a la estación de carga (+2) y no a la zona de entrega (+10), porque queda más cerca y el camino es más seguro.

**2.** Una recompensa menor puede ser óptima porque lo que se maximiza es la recompensa esperada descontando el costo de cada paso (-1) y el riesgo de caer en celdas de peligro. Llegar a +10 implica un camino más largo y más cercano a la zona de peligro mortal (-10), así que su valor neto termina siendo menor que ir directo a la carga +2.

**3.** El piso resbaloso cambia la decisión en las celdas (1,2), (2,1) y (3,3): ahí la política evita apuntar hacia direcciones donde una desviación (20% izquierda o derecha) podría llevar al robot cerca de una celda de peligro (-3 o -10), prefiriendo una ruta más segura aunque no sea la más corta.

**4.** El costo por paso -1 obliga al robot a preferir caminos cortos: sin ese costo, le daría igual demorarse muchos pasos, porque solo importaría a dónde llega y no cuánto le cuesta llegar.

**5.** Porque ahora la probabilidad de éxito de una acción depende del tipo de piso del estado actual (normal o resbaloso), no solo de la acción elegida. Por eso T(s,a,s') ya no es una única distribución fija para todos los estados, sino que cambia según si el estado está en slippery_states o no.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

**Predicción:** al bajar el costo por paso a -0.1, caminar deja de ser costoso, así que espero que el robot ya no tenga apuro y prefiera ir por la entrega +10 en vez de conformarse con la carga +2.

**Resultado:** correcto, el robot ahora va hacia la entrega +10 en vez de la carga +2. Con costo por paso casi nulo, ya no hay prisa por terminar rápido, así que compensa el camino más largo para conseguir la recompensa mayor.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

**Predicción:** al bajar la probabilidad de acierto en piso resbaloso de 0.60 a 0.40, espero que la política se vuelva más cautelosa cerca de las celdas resbalosas, aunque el destino final desde START probablemente no cambie.

**Resultado:** el destino desde START no cambió (sigue yendo a la carga +2) y la política en las celdas resbalosas tampoco cambió de dirección. Lo que sí baja son los valores V en esas celdas y alrededor, porque ahora hay más probabilidad de desviarse. La ruta elegida es lo bastante segura como para aguantar más resbalón sin cambiar de decisión.


### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

**Predicción:** al subir gamma a 0.99, el robot va a valorar casi igual una recompensa lejana que una cercana, así que espero que ahora sí prefiera ir por la entrega +10.

**Resultado:** correcto, con gamma=0.99 el robot cambia de destino y va hacia la entrega +10. Al descontar menos el futuro, una recompensa lejana pesa casi igual que una cercana, así que compensa el viaje más largo.

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [7]:
prev_dest = None
for lr in [x/100 for x in range(0, -1000, -1)]:
    grid.living_reward = lr
    V2, _ = value_iteration(grid)
    pi2 = extract_policy(grid, V2)
    s = grid.start
    for _ in range(30):
        if grid.is_terminal(s):
            break
        s = grid._next_state(s, pi2[s])
    if prev_dest is not None and s != prev_dest:
        print(f"Cambio detectado entre living_reward={lr+0.01:.2f} (destino {prev_dest}) y living_reward={lr:.2f} (destino {s})")
        break
    prev_dest = s

grid.living_reward = -1.0  # lo devolvemos al valor original

Cambio detectado entre living_reward=-0.79 (destino (0, 5)) y living_reward=-0.80 (destino (2, 2))


**Conclusión del bonus:** el umbral aproximado está en living_reward ≈ -0.80. Con costos por paso menos severos que ese (por ejemplo -0.79 o mayor, más cercano a 0), el robot prefiere ir por la entrega +10. Con costos iguales o más severos que -0.80 (más negativos), prefiere conformarse con la carga +2, más cercana.